# 02 - Baseline models

Four baselines, all compared on the **validation** set only:

| # | Model | Imbalance handling |
|---|---|---|
| 1 | Logistic Regression | class weights |
| 2 | Logistic Regression | SMOTE |
| 3 | Random Forest | class weights |
| 4 | Random Forest | SMOTE |

SMOTE is applied to the training set only (fraud raised to 10% of legit, not 50/50, because only
331 real frauds exist). Each model is evaluated at the default 0.5 threshold and at the threshold that
maximises F1 on validation.

In [ ]:
import os
# the notebooks live in notebooks/, but all paths below are from the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Training the Random Forests takes a few minutes. Set to True to re-run everything.
RERUN = False
if RERUN:
    import runpy
    runpy.run_path("src/baselines.py")

res = pd.read_csv("reports/baseline_results.csv")
pd.set_option("display.width", 200, "display.max_columns", None)
res.set_index("model").T

## Main comparison

In [ ]:
cols = ["model", "PR_AUC", "ROC_AUC", "precision@best", "recall@best", "F1@best", "FP@best", "FN@best", "latency_ms"]
res[cols].round(3)

## Precision-recall curves (validation)

In [ ]:
display(Image("reports/figures/baseline_pr_curves.png"))

## What the results show

| Model | PR-AUC | ROC-AUC | F1 (tuned thr.) |
|---|---|---|---|
| LogReg + class weight | 0.699 | 0.966 | 0.836 |
| LogReg + SMOTE | 0.690 | 0.969 | 0.806 |
| **Random Forest + class weight** | **0.855** | 0.947 | **0.868** |
| Random Forest + SMOTE | 0.844 | 0.966 | 0.862 |

1. **Random Forest beats Logistic Regression by about 0.15 PR-AUC**, so the relationship is non-linear (as the
   EDA suggested). This is the baseline to beat: PR-AUC 0.855.
2. **Class weights vs SMOTE:** the difference (about 0.01) is within noise because validation only has 71 frauds.
   Class weights are simpler and faster (RF fit 94 s vs 139 s), so I use them from here on.
3. **Threshold matters.** Logistic Regression with class weights at the default 0.5 threshold has precision 0.05:
   1,197 normal customers flagged to catch 63 frauds. At the tuned threshold the same model has 7 false positives.
4. **ROC-AUC is misleading here.** Random Forest has the lowest ROC-AUC (0.947) but the highest PR-AUC (0.855).
   With 0.17% fraud, ROC-AUC is dominated by the huge legit class, which is why PR-AUC is the main metric.
5. The best threshold for LogReg + class weights was 1.0 (at the edge), so its "best threshold" row is not reliable.
6. All four models end up with the same TP=56 and FN=15 at their best threshold, which suggests the same
   frauds are missed by every model (looked at in the evaluation report).
7. Latency per transaction: Logistic Regression about 2 ms, Random Forest about 115 ms.

**Baseline champion: Random Forest + class weight.**